# Gretil Sound Counts (Efficient Class-based Approach)

This notebook uses a class-based workflow to efficiently count sound patterns in the Gretil corpus using Zoekt prefiltering and indexing. Important outputs (gmsound*.csv) are kept in the main directory for git management.

In [ ]:
import os
import subprocess
from pathlib import Path
from typing import List
import pandas as pd
from tqdm import tqdm
from time import sleep, time
from glob import glob
from indic_transliteration.sanscript import transliterate, IAST, DEVANAGARI, ITRANS

# iast_text = "ātmānaṃ rathinaṃ viddhi śarīraṃ ratham eva ca"
# devanagari_text = transliterate(iast_text, IAST, DEVANAGARI)
# print(devanagari_text)

# Directory setup
PREFILTER_DIR = Path('gretil_prefiltered~')
INDEX_DIR = Path('gretil_zoekt_index~')
PREFILTER_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)

# Patterns
IAST_VOWEL_UNITS = [
    # Vowels
    'ai', 'au', 'ā', 'ī', 'ū', 'ṛ', 'ṝ', 'ḷ', 'ḹ', 'a', 'i', 'u', 'e', 'o',
]

IAST_CONSONANT_UNITS = [
    'k', 'kh', 'g', 'gh', 'ṅ', # Gutturals (ka-varga)
    'c', 'ch', 'j', 'jh', 'ñ', # Palatals (ca-varga)
    'ṭ', 'ṭh', 'ḍ', 'ḍh', 'ṇ', # Retroflexes (ṭa-varga)
    't', 'th', 'd', 'dh', 'n', # Dentals (ta-varga)
    'p', 'ph', 'b', 'bh', 'm', # Labials (pa-varga)
    'y', 'r', 'l', 'v',        # Semivowels
    'ś', 'ṣ', 's',             # Sibilants
    'h',                       # Aspirate
    'ṃ', 'ṁ', 'ḥ'              # Other
]

# VOWELS = list('aeiouAEIOU') + ['ā', 'ē', 'ī', 'ō', 'ū']
# VOWELS_SET = set(
# ALL_LETTERS = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZāēīōūṃṅñṭḍṇśṣḥ')
# CONSONANTS = sorted(list(ALL_LETTERS - VOWEL_SET))

VOWELS = IAST_VOWEL_UNITS
CONSONANTS = IAST_CONSONANT_UNITS

class ZoektPreFilter:
    def __init__(self, prefilter_dir: Path):
        self.prefilter_dir = prefilter_dir
        self.vowel_file = prefilter_dir / 'gretil_vowel_lines.txt'
        self.consonant_file = prefilter_dir / 'gretil_consonant_lines.txt'

    def filter_lines(self, force: bool = False):
        # Process vowel patterns first
        vowel_cmd = "zoekt 'm\\s+[aeiouAEIOUāēīōū]' file:\"^sa_.*.txt\""
        if self.vowel_file.exists() and not force:
            print(f"Vowel file {self.vowel_file} already exists. Skipping extraction.")
        else:
            print(f"{time():.2f} Running: {vowel_cmd} > {self.vowel_file}");
            subprocess.run(f"{vowel_cmd} > {self.vowel_file}", shell=True)

        # Process consonant patterns second
        consonant_cmd = "zoekt 'ṃ\\s+[^aeiouAEIOUāēīōū]' file:\"^sa_.*.txt\""
        if self.consonant_file.exists() and not force:
            print(f"Consonant file {self.consonant_file} already exists. Skipping extraction.")
        else:
            print(f"{time():.2f} Running: {consonant_cmd} > {self.consonant_file}");
            subprocess.run(f"{consonant_cmd} > {self.consonant_file}", shell=True)

        # Add file counts for verification
        if self.vowel_file.exists() and self.consonant_file.exists():
            with open(self.vowel_file, 'r') as f:
                vowel_lines = len(f.readlines())
            with open(self.consonant_file, 'r') as f:
                consonant_lines = len(f.readlines())
            print(f"\nExtracted {vowel_lines} vowel lines and {consonant_lines} consonant lines");

class ZoektIndexer:
    def __init__(self, src_dir: Path, index_dir: Path):
        self.src_dir = src_dir
        self.index_dir = index_dir

    def create_index(self, force: bool = False):
        # zoekt-index -file_limit 40000000 -max_trigram_count 2000000 -index ./gretil_zoekt_index~ ./gretil_prefiltered~
        cmd = f"zoekt-index -file_limit 40000000 -max_trigram_count 2000000 -index {self.index_dir} {self.src_dir}"
        if self.index_dir.exists() and len(glob(f"{self.index_dir}/*.zoekt"))>0 and not force:
            print(f"Index directory {self.index_dir} already exists. Skipping indexing.")
            return
        print(f"{cmd}\n\tIndexing {self.src_dir} into {self.index_dir}\n");
        subprocess.run(cmd, shell=True)

class PatternCounter:
    def __init__(self, index_dir: Path, patterns: List[str], output_csv: str):
        self.index_dir = index_dir
        self.patterns = patterns
        self.output_csv = output_csv

    def enum_patterns(self):
        acc = []
        for pat in tqdm(self.patterns, desc='Counting patterns'):
            cmd = f"zoekt -index_dir {self.index_dir} '{pat}'"
            # print(f"{time():.2f} Running: {cmd}");
            result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
            lines = [[pat] + line.split(':') for line in result.stdout.strip().split('\n') if line.strip()]
            if not lines:
#                 print(f"No matches found for pattern: {pat}")
                continue
            # Ensure we have at least 4 columns
            acc.append(lines)
        flatten_acc = [ [elem[s] for s in (0,3,-2,-1)] for sublist in acc for elem in sublist]
        df = pd.DataFrame(flatten_acc, columns=['pattern', 'kavya', 'line', 'text'])
        df.to_csv(self.output_csv)
        return df

    @classmethod
    def count_patterns(cls, index_dir: Path, force: bool = False) -> tuple:

        vc_df_name = 'gmsound_stacked.csv'
        try:
            if force : raise Exception("Redo Counts")
            print(f"Loading counts from {vc_df_name}")
            vc_df = pd.read_csv(vc_df_name, index_col=0)
        except Exception as e:
            print(f"Error reading {vc_df_name}: {e}")
            # Define patterns for vowels and consonants
            vowels = [rf"m\s+{v}" for v in VOWELS]
            consonants = [rf"ṃ\s+{v}" for v in CONSONANTS]
            v_df = cls(index_dir, vowels, 'vowel_counts.csv').enum_patterns()
            c_df = cls(index_dir, consonants, 'consonant_counts.csv').enum_patterns()
            vc_df = pd.concat([v_df, c_df])
            vc_df.to_csv(vc_df_name, index=False)

        # pivot the DataFrame to get counts per file per pattern
        pvt_df_name = 'gmsound_raw.csv'
        norm_df_name = 'gmsound_normalized.csv'
        try:
            if force : raise Exception("Redo Pivot")
            pvt_df = pd.read_csv(pvt_df_name, index_col=0)
            print(f"Loaded pivot table from {pvt_df_name}")
            return pvt_df
        except Exception as e:
            print(f"Error reading {pvt_df_name}: {e}")
            pvt_df = vc_df.pivot_table(index='kavya', columns='pattern', values='line', aggfunc='count', fill_value=0)
            # remove \s from column names
            pvt_df.columns = pvt_df.columns.str.replace(r'\\s\+', '_', regex=True)
            # remove ^sa_ and .txt$ from index
            pvt_df.index = pvt_df.index.str.replace(r'^sa_', '', regex=True).str.replace(r'\.txt$', '', regex=True)
            # remove any leading/trailing whitespace from index
            pvt_df.index = pvt_df.index.str.strip()
            pvt_df.to_csv(pvt_df_name, index=True)

            norm_df = pvt_df.div(pvt_df.sum(axis=1), axis=0)
            norm_df.to_csv(norm_df_name, index=True)
            print(f"Pivot table saved to {pvt_df_name}")
        return pvt_df, norm_df


# # Example usage (uncomment and run step by step):
# Step 1: Prefilter
prefilter = ZoektPreFilter(PREFILTER_DIR)
prefilter.filter_lines();


# Step 2: Index
indexer = ZoektIndexer(PREFILTER_DIR , INDEX_DIR);
indexer.create_index();


# Step 3: Count all patterns
df, norm_df = PatternCounter.count_patterns(INDEX_DIR, force=True)
display(df.head().T)
# Display normalized counts
display(norm_df.head().T)

Vowel file gretil_prefiltered~/gretil_vowel_lines.txt already exists. Skipping extraction.
Consonant file gretil_prefiltered~/gretil_consonant_lines.txt already exists. Skipping extraction.

Extracted 164467 vowel lines and 201254 consonant lines
Index directory gretil_zoekt_index~ already exists. Skipping indexing.
Error reading gmsound_stacked.csv: Redo Counts


Counting patterns: 100%|██████████| 36/36 [02:17<00:00,  3.81s/it]


Error reading gmsound_raw.csv: Redo Pivot
Pivot table saved to gmsound_raw.csv


kavya,108-buddhist-stotras,AGgirasasmRti,AdizeSa-paramArthasAra,AdyanAtha-anuttaraprakAzapaJcAzikA,AlambanaparIkSA
pattern,,,,,
m_a,10,62,18,7,0
m_ai,1,0,1,0,0
m_au,1,2,0,0,0
m_e,1,12,8,4,0
m_i,0,2,12,6,0
m_o,1,0,0,0,0
m_u,6,4,6,2,0
m_ā,9,15,5,2,0
m_ī,1,0,0,2,0


kavya,108-buddhist-stotras,AGgirasasmRti,AdizeSa-paramArthasAra,AdyanAtha-anuttaraprakAzapaJcAzikA,AlambanaparIkSA
pattern,,,,,
m_a,0.126582,0.029218,0.185567,0.152174,0.000000
m_ai,0.012658,0.000000,0.010309,0.000000,0.000000
m_au,0.012658,0.000943,0.000000,0.000000,0.000000
m_e,0.012658,0.005655,0.082474,0.086957,0.000000
m_i,0.000000,0.000943,0.123711,0.130435,0.000000
m_o,0.012658,0.000000,0.000000,0.000000,0.000000
m_u,0.075949,0.001885,0.061856,0.043478,0.000000
m_ā,0.113924,0.007069,0.051546,0.043478,0.000000
m_ī,0.012658,0.000000,0.000000,0.043478,0.000000
